# Query Agent: Contract Analysis with Conversational Memory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saskinosie/weaviate_query_agent_demo/blob/main/query_agent_contract_agent.ipynb)

This notebook demonstrates advanced **contract analysis capabilities** using the Weaviate Query Agent with **conversational memory**. Unlike traditional contract review tools, this system maintains context across multiple queries, enabling sophisticated multi-step analysis workflows.

## Overview

The Query Agent transforms contract analysis by:
- **Contextual Understanding**: Maintains conversation history for complex analysis workflows
- **Risk Assessment**: Automatically identifies and categorizes contract risks
- **Comparative Analysis**: Benchmarks contract terms across your portfolio
- **Strategic Recommendations**: Provides actionable business guidance
- **Template Generation**: Creates protective clauses based on real contract analysis

## Dataset

This demonstration uses a financial contracts dataset containing:
- **Partnership Agreements**: Collaboration terms, profit sharing, and joint venture structures
- **Loan Agreements**: Lending terms, interest rates, and repayment schedules  
- **Service Agreements**: Professional services, termination clauses, and liability terms
- **Employment Contracts**: Compensation, termination, and non-compete provisions

All contracts are pre-vectorized with **Snowflake Arctic Embed v2.0** embeddings for semantic search capabilities.

## Key Features Demonstrated

1. **Conversational Memory**: Each query builds on previous analysis
2. **Multi-Step Workflows**: From portfolio overview → risk analysis → strategic recommendations
3. **Template Generation**: Creates actionable contract templates from analysis
4. **Rapid Follow-up Questions**: Context-aware queries without repeating information

## Business Applications

- **Due Diligence**: Comprehensive contract portfolio analysis
- **Risk Management**: Identify and prioritize contract vulnerabilities
- **Negotiation Strategy**: Benchmark terms and identify leverage points
- **Compliance Auditing**: Systematic review of contract terms and obligations
- **Template Creation**: Generate protective clauses based on portfolio insights

<a id='TOC'></a>  
## Table of Contents  

1. <a href="#Dependencies">Dependencies</a><br>
2. <a href="#Connecting">Connecting to your Weaviate cluster</a><br>
3. <a href="#Collections">Setting up collections and loading contract data</a><br>
4. <a href="#QueryAgent">Creating the Query Agent</a><br>
5. <a href="#ConversationalAnalysis">Conversational Contract Analysis with Memory</a><br>
   5.1 <a href="#Step1">Step 1: Initialize Contract Analysis Conversation</a><br>
   5.2 <a href="#Step2">Step 2: Deep Dive Risk Analysis</a><br>
   5.3 <a href="#Step3">Step 3: Comparative Benchmarking</a><br>
   5.4 <a href="#Step4">Step 4: Strategic Recommendations</a><br>
   5.5 <a href="#Step5">Step 5: Template Creation for Future Contracts</a><br>
6. <a href="#MemoryDemo">Conversation Memory in Action</a><br>
   6.1 <a href="#QuickQuestions">Quick Follow-up Questions</a><br>
   6.2 <a href="#Scenarios">Advanced Follow-up: What If Scenarios</a><br>
   6.3 <a href="#MemorySummary">Memory Summary</a><br>

<a id='Dependencies'></a>  
## 1. Dependencies
[Back to table of contents](#TOC)

This section initializes the necessary dependencies and sets environment variables for connecting to your Weaviate Cloud instance.

**If using Colab, add your keys between the quotation marks in the cell below**

In [1]:
# un this cell to load your keys in Colab
import os

# set environment variables
os.environ['WEAVIATE_URL'] = ''  # weaviate instance url
os.environ['WEAVIATE_API_KEY'] = ''  # admin api key

**If running locally, make sure your keys are in your .env file and run this cell**

In [2]:
# Run this cell if you are working locally in VS code or Code Space
import dotenv

dotenv.load_dotenv(override=True)

True

<a id='Connecting'></a>  
## 2. Connecting to your Weaviate cluster  
[Back to table of contents](#TOC)  

To interact with our Weaviate cluster, we'll initialize a client object and verify the connection is successful. This connection will be used throughout the notebook for all data operations and Query Agent interactions.

In [3]:
import weaviate, os

# Connect to Weaviate Cloud
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.getenv("WEAVIATE_URL"),
    auth_credentials=weaviate.auth.AuthApiKey(os.getenv("WEAVIATE_API_KEY"))
)

# check if the connection is successful
client.is_ready()

True

<a id='Collections'></a>  
## 3. Setting up collections and loading contract data  
[Back to table of contents](#TOC)  

In this section, we'll create the Financialcontracts collection and populate it with pre-vectorized financial contract data from the [Weaviate Agents dataset on Hugging Face](https://huggingface.co/datasets/weaviate/agents). The collection uses the Snowflake Arctic Embed v2.0 model for semantic search capabilities.

In [4]:
# import required module for configuration
import weaviate.classes.config as wc

# delete collection if it exists
if client.collections.exists("Financialcontracts"):
    client.collections.delete("Financialcontracts")

client.collections.create(
    name="Financialcontracts", #Collection name

    #Set vectorizer configuration for text2vec using Weaviates embedding model
    vector_config=wc.Configure.Vectors.text2vec_weaviate(
        model="Snowflake/snowflake-arctic-embed-l-v2.0",
        source_properties=["contract_text"],
    ),
)

print("Successfully created collection: Financialcontracts.")

Successfully created collection: Financialcontracts.


In [5]:
# Upload brands data using the Weaviate-recommended streaming approach
from datasets import load_dataset

# Upload books data using the Weaviate-recommended streaming approach
books_dataset = load_dataset("weaviate/agents", "query-agent-financial-contracts", split="train", streaming=True)

books_collection = client.collections.get("Financialcontracts")
with books_collection.batch.fixed_size(batch_size=100) as batch:
    for item in books_dataset:
        batch.add_object(
            properties=item["properties"],
            vector=item["vector"]
        )

print("Successfully uploaded Financialcontracts data using dataset streaming method!")

Successfully uploaded Financialcontracts data using dataset streaming method!


<a id='QueryAgent'></a>  
## 4. Creating the Query Agent  
[Back to table of contents](#TOC)  

Now we'll create our Query Agent instance, which will intelligently route queries across our three collections (Brands, Ecommerce, Books) and automatically determine the optimal search strategy.

The output from the Query Agent will include: the original query, the generated answer to the query, the searches performed, the collections searched, filters applied, aggregates completed, source objects pulled from the database that comntributed to the generated answer and any mising information if the generated answer is incomplete.

In [ ]:

from weaviate.agents.query import QueryAgent

# Instantiate a new agent object, and specify the collections to query
qa = QueryAgent(
    client=client,
    collections=["Brands", "Ecommerce", "Books", "Financialcontracts"],
)

# Import ChatMessage for conversational memory functionality
from weaviate.agents.classes import ChatMessage
print("✓ ChatMessage imported - ready for conversational contract analysis!")

<a id='ContractAnalysis'></a>  
## Practical Business Use Case for the Query Agent: Contract Analysis  
[Back to table of contents](#TOC)  

This notebook demonstrates practical business applications of the Query Agent through contract analysis scenarios. We'll explore how the agent can be used for risk assessment, contract comparison, and drafting suggestions in real-world business contexts.

<a id='ConversationalAnalysis'></a>  
# 5. Conversational Contract Analysis with Memory
[Back to table of contents](#TOC)

Now we'll demonstrate how the Query Agent can maintain conversation context, allowing for natural follow-up questions and progressive analysis that builds on previous findings.

<a id='Step1'></a>  
### 5.1 Step 1: Initialize Contract Analysis Conversation   
[Back to table of contents](#TOC)

We'll start a conversation with the Query Agent to analyze our contract portfolio. Each subsequent query will build on previous context.

In [ ]:
# Initialize our conversation with context setting
conversation = [
    ChatMessage(role="user", content="Hello! I need help analyzing our contract portfolio."),
    ChatMessage(role="assistant", content="Hello! I can help you analyze your contract portfolio. I have access to your financial contracts database. What would you like to explore?"),
    ChatMessage(role="user", content="Let's start with an overview - what types of contracts do we have and who are our most frequent contracting parties?")
]

# First query - Portfolio Discovery
response = qa.ask(conversation)
print("PORTFOLIO OVERVIEW:")
print(response.final_answer)
print("\n" + "="*50)

# Add response to conversation memory
conversation.append(ChatMessage(role="assistant", content=response.final_answer))

🔍 PORTFOLIO OVERVIEW:
Your contract portfolio consists of the following main types of contracts:

1. Employment Contracts
2. Loan Agreements
3. Lease Agreements
4. Service Agreements
5. Partnership Agreements
6. Sales Agreements
7. Invoice Contracts

Regarding the most frequent contracting parties (authors), the main individuals or entities involved include:

- Authors like Hans Zimmer, Arthur Penndragon, Bob Brown, Alice Johnson, Johnathan Smith, and John Williams.
- Frequent contracting parties such as Mark Robson, John Smith, Danny Williams, and the company Weaviate itself.
- Mark Robson appears multiple times in lease, employment, and partnership agreements.
- Weaviate is consistently the party either providing or receiving services or leases.

Overall, leases and agreements involving Weaviate and Mark Robson are predominant, with other contracts involving various individuals and authors responsible for drafting. If you want, I can provide more detailed counts or further breakdowns

### Step 2: Deep Dive Risk Analysis 
[Back to table of contents](#TOC)

Now we'll drill down on specific findings from our portfolio overview. The agent will remember the previous context and can reference specific parties or contract types we discovered.

In [ ]:
# Follow-up query building on previous context
# You can modify these parameters based on the findings from Step 1
high_frequency_party = "OpenAI"  # Adjust based on Step 1 results
risk_focus = "termination clauses"

conversation.append(ChatMessage(
    role="user", 
    content=f"Based on our portfolio overview, I'm concerned about {high_frequency_party} since they appear in multiple contracts. Can you examine all agreements with {high_frequency_party} and identify potential risks in their {risk_focus}?"
))

response = qa.ask(conversation)
print(f"RISK ANALYSIS - {high_frequency_party.upper()}:")
print(response.final_answer)
print("\n" + "="*50)

# Add to conversation memory
conversation.append(ChatMessage(role="assistant", content=response.final_answer))

⚠️ RISK ANALYSIS - OPENAI:
There are multiple partnership agreements between Weaviate and OpenAI spanning from 2022 through late 2023. These agreements share many common points regarding termination clauses:

- All agreements have a standard termination clause allowing either party to terminate the agreement by providing a written notice.
- The notice period specified is uniformly 30 days.
- Termination can be initiated without cause, simply by providing the 30 days' written notice.
- One of the latest agreements (dated October 2023) adds that termination can specifically occur if a party fails to comply with the terms and conditions, also requiring a 30 days' written notice.
- Upon termination, agreements specify that outstanding financial obligations or contributions are to be settled or reconciled.
- The agreements generally do not specify any penalties, cure periods beyond the notice, or conditions that could complicate termination.
  
Potential risks based on these termination ter

### Step 3: Comparative Benchmarking
[Back to table of contents](#TOC)

Now we'll compare our findings against the broader portfolio to see if the risks we identified are typical or outliers.

In [ ]:
# Benchmarking query that references previous analysis
conversation.append(ChatMessage(
    role="user",
    content="Now I want to benchmark these findings. Compare the termination and payment terms we just analyzed against our other partnership agreements. Are we getting fair terms compared to our other partners?"
))

response = qa.ask(conversation)
print("BENCHMARKING ANALYSIS:")
print(response.final_answer)
print("\n" + "="*50)

# Add to conversation memory
conversation.append(ChatMessage(role="assistant", content=response.final_answer))

📊 BENCHMARKING ANALYSIS:
Comparing the termination and payment terms between partnership agreements with OpenAI and other partners based on available contracts, here are the key points:

1. Termination Clauses:
- The partnership agreements with OpenAI consistently allow either party to terminate with a written notice of 30 days.
- The termination is typically permitted for failure to comply with obligations or for convenience, with no additional cure periods or penalties detailed.
- Other partnership agreements (including some with OpenAI) also commonly have the same 30-day written notice for termination upon non-compliance or as convenience, showing a consistent approach across partners.
- None of the reviewed partnership contracts with other partners includes termination terms more favorable than the 30-day notice; this is a standard term in the portfolio.

2. Payment and Financial Terms:
- OpenAI partnership agreements specify contributions in the range of approximately $173 to $726

### Step 4: Strategic Recommendations
[Back to table of contents](#TOC)

Based on our analysis and benchmarking, we'll now ask for actionable business recommendations.

In [ ]:
# Strategic recommendations based on all previous analysis
conversation.append(ChatMessage(
    role="user",
    content="""Based on our portfolio analysis, risk assessment, and benchmarking:
    1. Should we renegotiate with the parties we identified as risky?
    2. Which specific clauses need improvement?
    3. What leverage do we have in these negotiations?
    Please provide actionable recommendations."""
))

response = qa.ask(conversation)
print("STRATEGIC RECOMMENDATIONS:")
print(response.final_answer)
print("\n" + "="*50)

# Add to conversation memory
conversation.append(ChatMessage(role="assistant", content=response.final_answer))

🎯 STRATEGIC RECOMMENDATIONS:
Based on the portfolio analysis, risk assessment, and benchmarking, here are actionable recommendations addressing your three questions:

1. **Should you renegotiate with parties identified as risky?**  
   Yes. While the termination clauses are generally balanced and consistent across partners, the ease of termination with only 30 days’ notice and lack of cure periods or detailed dispute resolution poses operational and financial risks, especially with key partners like OpenAI and critical employees such as Mark Robson. Renegotiation with these parties is advisable to introduce more robust protections.

2. **Specific clauses needing improvement:**  
   - **Termination Clauses:** Extend the notice period beyond 30 days for key contracts to allow smoother transitions. Include explicit cure periods allowing a party to remedy breaches before termination. Add clearer procedures for termination for cause to prevent abrupt disruptions.  
   - **Payment and Financ

### Step 5: Template Creation for Future Contracts
[Back to table of contents](#TOC)

Finally, we'll generate a protective contract template based on all our findings to prevent future risks.

In [ ]:
# Template generation based on entire conversation context
conversation.append(ChatMessage(
    role="user",
    content="""Using all the insights from our portfolio analysis, risk assessment, benchmarking, and recommendations, create a 'Partnership Agreement Checklist' with 5 must-have protective clauses. Base this on the real issues we've identified in our contracts."""
))

response = qa.ask(conversation)
print("PROTECTIVE CONTRACT TEMPLATE:")
print(response.final_answer)
print("\n" + "="*50)

print("ANALYSIS COMPLETE!")
print("We've conducted a full contract portfolio analysis using conversational memory.")

📋 PROTECTIVE CONTRACT TEMPLATE:
Here is a tailored "Partnership Agreement Checklist" featuring 5 must-have protective clauses, designed to address the key risks and gaps identified in your contract portfolio analysis:

**Partnership Agreement Checklist: 5 Must-Have Protective Clauses**

1. **Extended Termination Notice & Cure Period**  
   - Require a minimum 60-day written notice for termination to allow orderly transition.  
   - Include a cure period (e.g., 30 days) after notice of breach before termination is effective, enabling remediation.

2. **Detailed Financial Settlement & Payment Terms**  
   - Clearly define procedures and deadlines for settling outstanding payments upon termination.  
   - Include dispute resolution methods specifically for financial disagreements post-termination.

3. **Termination for Cause with Specific Criteria**  
   - Specify clear grounds for termination for cause (e.g., material breach, non-compliance).  
   - Outline process for documenting and no

<a id='MemoryDemo'></a>  
# 6. Conversation Memory in Action
[Back to table of contents](#TOC)

This section demonstrates the power of conversational memory. Notice how we can ask follow-up questions without repeating any context - the agent remembers our entire analysis journey.

### Quick Follow-up Questions

These short queries work because the agent remembers our complete analysis. No need to repeat context!

In [ ]:
# Rapid-fire questions that leverage our conversation memory
quick_questions = [
    "What was the most concerning risk we found?",
    "Which party should we prioritize for renegotiation?", 
    "Can you summarize our 3 biggest vulnerabilities?",
    "What would happen if we don't act on these findings?"
]

print("RAPID-FIRE Q&A (No Context Needed!):")
print("="*60)

for i, question in enumerate(quick_questions, 1):
    conversation.append(ChatMessage(role="user", content=question))
    response = qa.ask(conversation)
    
    print(f"\nQ{i}: {question}")
    print(f"A{i}: {response.final_answer}")
    print("-" * 40)
    
    # Add response to memory for next question
    conversation.append(ChatMessage(role="assistant", content=response.final_answer))

🔥 RAPID-FIRE Q&A (No Context Needed!):

💬 Q1: What was the most concerning risk we found?
🤖 A1: The most concerning risk identified in your contract portfolio is the ease with which either party can terminate partnership agreements—particularly those with OpenAI—by providing only a 30-day written notice without any cure period or detailed dispute resolution mechanisms. This short notice period and lack of protections could lead to abrupt disruptions in ongoing collaborations and potential financial disputes due to insufficient time to properly transition or settle obligations.
----------------------------------------

💬 Q2: Which party should we prioritize for renegotiation?
🤖 A2: You should prioritize renegotiation with **OpenAI**. They appear in multiple partnership agreements with consistent 30-day termination clauses and relatively straightforward financial terms. The key risk of short termination notice, lack of cure periods, and ambiguous financial settlement procedures is most p

### Advanced Follow-up: What If Scenarios

Let's explore hypothetical scenarios based on our analysis findings.

In [ ]:
# Complex scenario planning using conversation context
conversation.append(ChatMessage(
    role="user",
    content="If we tried to renegotiate the risky contracts we identified, but the other parties refused our terms, what alternative strategies could we pursue to protect ourselves?"
))

response = qa.ask(conversation)
print("SCENARIO PLANNING:")
print(response.final_answer)
print("\n" + "="*50)

# One more advanced question
conversation.append(ChatMessage(role="assistant", content=response.final_answer))
conversation.append(ChatMessage(
    role="user",
    content="Based on everything we've discussed, if you were our Chief Legal Officer, what would be your top priority action item for next quarter?"
))

response = qa.ask(conversation)
print("EXECUTIVE RECOMMENDATION:")
print(response.final_answer)

conversation.append(ChatMessage(role="assistant", content=response.final_answer))

🎭 SCENARIO PLANNING:
If renegotiation attempts on the risky contracts, such as those with OpenAI, are refused, you can pursue several alternative strategies to protect your interests:

1. **Strengthen Internal Risk Management:**  
   - Develop contingency plans for abrupt contract terminations, including backup partners or alternative service providers.  
   - Increase operational flexibility to quickly adapt if key agreements end unexpectedly.

2. **Enhance Monitoring and Early Warning Systems:**  
   - Implement rigorous contract performance tracking to detect early signs of breaches or dissatisfaction.  
   - Maintain frequent communication with partners to address issues proactively before they escalate.

3. **Leverage Contractual Renewal Opportunities:**  
   - Use upcoming contract renewals as leverage to introduce improved terms gradually, even if immediate renegotiation fails.

4. **Diversify Partnerships:**  
   - Reduce dependency on any single partner by cultivating addition

### Memory Summary

Notice how our conversation evolved from simple questions to strategic planning - all while maintaining context throughout the entire journey!

In [16]:
# close the Weaviate client to free up resources
client.close()